<a href="https://colab.research.google.com/github/evapatel123/Build-Your-Own-ChatBot/blob/main/Easy_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio transformers accelerate

In [1]:
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread

# 1. Load the model and tokenizer directly onto the Colab GPU
# Using Qwen2.5-1.5B-Instruct so it fits perfectly and runs fast on Colab's free T4 GPU
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto" # Automatically moves the model to the GPU
)

# BULLETPROOF MEMORY FIX: We bypass Gradio's history system entirely using a clean Python list
chat_memory = [{"role": "system", "content": "You are a friendly chatbot."}]

def respond(message, history):
    global chat_memory

    # Reset memory if the user clears the chat interface
    if not history:
        chat_memory = [{"role": "system", "content": "You are a friendly chatbot."}]

    # Append the new user prompt directly to our clean list
    chat_memory.append({"role": "user", "content": message})

    # Keep the history length trimmed to prevent crashing out on long sessions
    if len(chat_memory) > 10:
        chat_memory = [chat_memory[0]] + chat_memory[-8:]

    # 3. Prepare the inputs for the model
    text = tokenizer.apply_chat_template(
        chat_memory,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 4. Set up a streamer so the response types out live (optional but recommended!)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    generation_kwargs = dict(
        model_inputs,
        streamer=streamer,
        max_new_tokens=512, #NOTE : If your internet is lagging, make sure to reduce the amount of tokens being used (512 is recommended)
        temperature=0.8
    )

    # Run the generation in a separate thread so it doesn't freeze the UI
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    # 5. Yield the text back to Gradio as it generates
    partial_text = ""
    for new_text in streamer:
        partial_text += new_text
        yield partial_text

    # After the bot finishes typing, log its response into the session memory safely
    chat_memory.append({"role": "assistant", "content": partial_text})

# 6. Launch the interface with sharing turned on
# share=True creates a public link you can share outside of Colab
chatbot = gr.ChatInterface(respond)
chatbot.launch(share=True)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aca2fc5daba555d727.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
